# Notebook 08: Reporte de Calidad Completo

**Duración**: 30 minutos | **Nivel**: Intermedio

## Introducción

Integramos todas las dimensiones de calidad en un reporte maestro.

### Objetivos:
1. Crear suite maestra con todas las dimensiones
2. Generar scorecard de calidad
3. Crear dashboard completo con Data Docs

In [ ]:
import great_expectations as gx
import pandas as pd
from datetime import datetime

df = pd.read_csv("../data/ventas_sucias.csv")
print(f"Dataset: {len(df)} registros, {len(df.columns)} columnas")

## Suite Maestra de Calidad

In [ ]:
context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

# Suite maestra
suite = context.suites.add(gx.ExpectationSuite(name="calidad_completa"))

# COMPLETITUD
for col in ["order_id", "customer_id", "price", "quantity"]:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(
            column=col, meta={"dimension": "Completitud"}
        )
    )

# VALIDEZ
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price", min_value=0.01, max_value=10000,
        meta={"dimension": "Validez"}
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="quantity", min_value=1, max_value=100,
        meta={"dimension": "Validez"}
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="product_category",
        value_set=["Electronics", "Clothing", "Home", "Toys"],
        meta={"dimension": "Validez"}
    )
)

# CONSISTENCIA
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="price", type_="float64",
        meta={"dimension": "Consistencia"}
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="order_id",
        regex=r"^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$",
        meta={"dimension": "Consistencia"}
    )
)

# UNICIDAD
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="order_id", meta={"dimension": "Unicidad"}
    )
)

# PUNTUALIDAD
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_date",
        min_value="2020-01-01",
        max_value=datetime.now().strftime("%Y-%m-%d"),
        meta={"dimension": "Puntualidad"}
    )
)

suite.save()
print(f" Suite maestra creada con {len(suite.expectations)} expectativas")

## Ejecutar Validación Completa

In [ ]:
val_def = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite, name="validacion_completa")
)

resultado = val_def.run(batch_parameters={"dataframe": df})

print("\n" + "=" * 70)
print("SCORECARD DE CALIDAD DE DATOS")
print("=" * 70)
print(f"\nEstado General: {' APROBADO' if resultado.success else ' RECHAZADO'}")
print(f"\nTotal Expectativas: {len(resultado.results)}")
print(f"Exitosas: {sum(1 for r in resultado.results if r.success)}")
print(f"Fallidas: {sum(1 for r in resultado.results if not r.success)}")
print(f"Tasa de Éxito: {(sum(1 for r in resultado.results if r.success) / len(resultado.results) * 100):.1f}%")

# Por dimensión
print("\n" + "=" * 70)
print("RESULTADOS POR DIMENSIÓN")
print("=" * 70)

dimensiones = {}
for result in resultado.results:
    dim = result.expectation_config.meta.get('dimension', 'Sin clasificar')
    if dim not in dimensiones:
        dimensiones[dim] = {'total': 0, 'exitosas': 0}
    dimensiones[dim]['total'] += 1
    if result.success:
        dimensiones[dim]['exitosas'] += 1

for dim, stats in dimensiones.items():
    tasa = (stats['exitosas'] / stats['total'] * 100)
    emoji = "" if tasa == 100 else "⚠️" if tasa >= 80 else ""
    print(f"\n{emoji} {dim}:")
    print(f"   Exitosas: {stats['exitosas']}/{stats['total']} ({tasa:.1f}%)")

## Generar Dashboard Completo

In [ ]:
context.build_data_docs()

print("\n" + "=" * 70)
print(" DASHBOARD DE CALIDAD GENERADO")
print("=" * 70)
print("\nEl Data Doc incluye:")
print("  Scorecard general de calidad")
print("  Resultados por dimensión")
print("  Detalles de cada expectativa")
print("  Gráficos interactivos")
print("  Ejemplos de datos problemáticos")

context.open_data_docs()

##  Resumen del Módulo 2

Has completado el estudio de las **6 Dimensiones de Calidad**:

1.  **Completitud**: Datos presentes
2.  **Validez**: Rangos y formatos correctos
3.  **Consistencia**: Coherencia entre campos
4.  **Unicidad**: Sin duplicados
5.  **Puntualidad**: Datos actualizados
